# Genetic Engineering In Silico

## Tier 3 - Applied Bioinformatics

This notebook implements core molecular cloning and genome-editing workflows entirely in Python.
All algorithms are built from first principles — no external bioinformatics libraries required.

## Learning Objectives

By the end of this notebook you will be able to:

1. Simulate restriction digestion and predict gel electrophoresis banding patterns
2. Design PCR primers using Tm models ranging from the 4+2 rule to nearest-neighbor thermodynamics
3. Evaluate primer quality: GC content, 3' runs, dimer formation, and hairpin potential
4. Model traditional RE-based cloning and verify reading-frame continuity after ligation
5. Identify CRISPR guide RNA candidates for SpCas9, SaCas9, and Cas12a and rank them by on-target score
6. Optimize a coding sequence for heterologous expression using organism-specific codon usage tables
7. Design Gibson Assembly overlaps for multi-fragment constructs

---

[< Previous: Biochemistry and Enzyme Kinetics](../13_Biochemistry_and_Enzyme_Kinetics/02_enzyme_kinetics.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Population Genetics >](../15_Population_Genetics_and_Molecular_Evolution/01_population_genetics.ipynb)

---

## 1. In Silico Restriction Digestion

### 1.1 Restriction Enzymes: Recognition Sites and Cut Geometry

Restriction endonucleases recognise specific short DNA sequences (typically 4–8 bp) and cleave both strands.
The recognition sequences of Type II enzymes — the workhorses of cloning — are **palindromic**: the sequence reads the same 5'→3' on both strands.

Cut geometry determines the overhang type:

| Overhang | Example | Description |
|----------|---------|-------------|
| 5' overhang (sticky) | EcoRI | Top strand cut upstream of bottom strand |
| 3' overhang (sticky) | KpnI  | Top strand cut downstream of bottom strand |
| Blunt | SmaI | Both strands cut at the same position |

We represent each enzyme as a recognition sequence plus two cut positions: the position on the **top strand** and the position on the **bottom strand** (both counted from the start of the recognition sequence, 0-indexed).

In [ ]:
import re
import math
import itertools
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Restriction enzyme database
# Each entry: (recognition_seq, cut_top, cut_bottom)
# cut positions are 0-indexed from start of recognition site on top strand
# cut_bottom is on the complementary strand read 5'->3' (i.e. counted from end of site)
RESTRICTION_ENZYMES = {
    'EcoRI':  ('GAATTC', 1, 5),   # 5' AATT overhang
    'BamHI':  ('GGATCC', 1, 5),   # 5' GATC overhang
    'HindIII':('AAGCTT', 1, 5),   # 5' AGCT overhang
    'SalI':   ('GTCGAC', 1, 5),   # 5' TCGA overhang
    'XhoI':   ('CTCGAG', 1, 5),   # 5' TCGA overhang (compatible with SalI)
    'NcoI':   ('CCATGG', 1, 5),   # 5' CATG overhang
    'NdeI':   ('CATATG', 2, 4),   # 5' AT overhang
    'SmaI':   ('CCCGGG', 3, 3),   # blunt
    'XmaI':   ('CCCGGG', 1, 5),   # 5' CCGG overhang (isoschizomer of SmaI, different cut)
    'KpnI':   ('GGTACC', 5, 1),   # 3' GTAC overhang
    'SacI':   ('GAGCTC', 5, 1),   # 3' AGCT overhang
    'NotI':   ('GCGGCCGC', 2, 6), # 5' GGCC overhang
    'XbaI':   ('TCTAGA', 1, 5),   # 5' CTAG overhang
    'SpeI':   ('ACTAGT', 1, 5),   # 5' CTAG overhang (compatible with XbaI, scar site)
    'PstI':   ('CTGCAG', 5, 1),   # 3' TGCA overhang
    'ClaI':   ('ATCGAT', 2, 4),   # 5' CG overhang
    'AvaI':   ('CYCGRG', 1, 5),   # degenerate; C[CT]CG[AG]G
}

def complement(base: str) -> str:
    return {'A':'T','T':'A','G':'C','C':'G','N':'N',
            'R':'Y','Y':'R','W':'W','S':'S','M':'K','K':'M'}.get(base, 'N')

def reverse_complement(seq: str) -> str:
    return ''.join(complement(b) for b in reversed(seq.upper()))

def iupac_to_regex(seq: str) -> str:
    iupac = {'A':'A','T':'T','G':'G','C':'C','N':'[ATGC]',
              'R':'[AG]','Y':'[CT]','W':'[AT]','S':'[GC]',
              'M':'[AC]','K':'[GT]','B':'[CGT]','D':'[AGT]',
              'H':'[ACT]','V':'[ACG]'}
    return ''.join(iupac.get(b, b) for b in seq.upper())

print('Restriction enzyme database loaded.')
print(f'Enzymes available: {len(RESTRICTION_ENZYMES)}')
for name, (site, ct, cb) in list(RESTRICTION_ENZYMES.items())[:5]:
    overhang_len = abs(ct - cb)
    overhang_type = '5-prime' if ct < cb else ('3-prime' if ct > cb else 'blunt')
    print(f'  {name:10s}  site={site}  cut_top={ct}  cut_bot={cb}  '
          f'overhang={overhang_len}bp {overhang_type}')

### 1.2 Single-Enzyme Digest: Finding Cut Sites and Computing Fragments

In [ ]:
def find_cut_positions(dna: str, enzyme_name: str) -> list[int]:
    """Return top-strand cut positions for all recognition sites of enzyme in dna."""
    site, cut_top, _ = RESTRICTION_ENZYMES[enzyme_name]
    pattern = iupac_to_regex(site)
    positions = []
    for m in re.finditer(f'(?={pattern})', dna.upper()):
        positions.append(m.start() + cut_top)
    return sorted(positions)

def digest(dna: str, enzyme_name: str, circular: bool = False) -> list[str]:
    """Return list of fragment sequences after digestion."""
    cuts = find_cut_positions(dna, enzyme_name)
    if not cuts:
        return [dna]
    if circular:
        # rotate so first cut is at position 0
        first = cuts[0]
        rotated = dna[first:] + dna[:first]
        adjusted = [c - first for c in cuts[1:]] + [len(dna)]
        fragments = []
        prev = 0
        for c in adjusted:
            fragments.append(rotated[prev:c])
            prev = c
        return fragments
    else:
        boundaries = [0] + cuts + [len(dna)]
        return [dna[boundaries[i]:boundaries[i+1]] for i in range(len(boundaries)-1)]

# Demo: digest a synthetic plasmid-like sequence
plasmid_seq = (
    'AAGCTTGCATGCCTGCAGGTCGACGGATCCCCGGAATTCGAGCTCGGTACCCGGGGATCCTCTAGAGTCGAC'
    'CTGCAGGCATGCAAGCTTGGCGTAATCATGGTCATAGCTGTTTCCTGTGTGAAATTGTTATCCGCTCACAATT'
    'CCACACAACATACGAGCCGGAAGCATAAAGTGTAAAGCCTGGGGTGCCTAATGAGTGAGCTAACTCACATTAAT'
    'TGCGTTGCGCTCACTGCCCGCTTTCCAGTCGGGAAACCTGTCGTGCCAGCTGCATTAATGAATCGGCCAACGCG'
    'CGGGGAGAGGCGGTTTGCGTATTGGGCGCTCTTCCGCTTCCTCGCTCACTGACTCGCTGCGCTCGGTCGTTCGG'
    'CTGCGGCGAGCGGTATCAGGCTACGGCGTAGCGCACGACCGGGCGAGCTCCTCAAGCTTGTGAATTCGGATCC'
)

for enzyme in ['EcoRI', 'BamHI', 'HindIII']:
    cuts = find_cut_positions(plasmid_seq, enzyme)
    frags = digest(plasmid_seq, enzyme)
    print(f'{enzyme}: {len(cuts)} cut(s) at positions {cuts}')
    print(f'  Fragments: {[len(f) for f in frags]} bp')

### 1.3 Multi-Enzyme Double Digest

In [ ]:
def double_digest(dna: str, enzyme1: str, enzyme2: str, circular: bool = False) -> list[str]:
    """Digest dna with two enzymes simultaneously."""
    cuts1 = find_cut_positions(dna, enzyme1)
    cuts2 = find_cut_positions(dna, enzyme2)
    all_cuts = sorted(set(cuts1 + cuts2))
    if not all_cuts:
        return [dna]
    if circular:
        first = all_cuts[0]
        rotated = dna[first:] + dna[:first]
        adjusted = [c - first for c in all_cuts[1:]] + [len(dna)]
        frags, prev = [], 0
        for c in adjusted:
            frags.append(rotated[prev:c])
            prev = c
        return frags
    boundaries = [0] + all_cuts + [len(dna)]
    return [dna[boundaries[i]:boundaries[i+1]] for i in range(len(boundaries)-1)]

frags_single = digest(plasmid_seq, 'EcoRI')
frags_double = double_digest(plasmid_seq, 'EcoRI', 'BamHI')

print('Single digest (EcoRI):')
print(f'  {sorted([len(f) for f in frags_single], reverse=True)} bp')
print('Double digest (EcoRI + BamHI):')
print(f'  {sorted([len(f) for f in frags_double], reverse=True)} bp')

### 1.4 Simulating Gel Electrophoresis

In agarose gel electrophoresis, DNA fragments migrate based on size. Migration distance is approximately proportional to the **log** of fragment length. We simulate a lane by plotting bands at log-scale positions.

In [ ]:
def plot_gel(lanes: dict[str, list[str]], ladder_sizes: list[int] | None = None):
    """Simulate an agarose gel image. lanes maps lane label to list of fragment sequences."""
    if ladder_sizes is None:
        ladder_sizes = [10000, 8000, 6000, 5000, 4000, 3000, 2000, 1500, 1000, 750, 500, 250, 100]

    all_lanes = {'Ladder': [str(s) for s in ladder_sizes]}
    all_lanes.update(lanes)
    n_lanes = len(all_lanes)

    fig, ax = plt.subplots(figsize=(2.5 * n_lanes, 5))
    ax.set_facecolor('#f5f0e8')   # agarose colour
    ax.set_ylim(1.8, 4.1)
    ax.set_xlim(0, n_lanes)
    ax.set_yticks([])
    ax.set_xticks([])

    for lane_idx, (label, frags) in enumerate(all_lanes.items()):
        x_center = lane_idx + 0.5
        sizes = [len(f) if not f.isdigit() else int(f) for f in frags]
        for size in sizes:
            if size < 50:
                continue
            y = math.log10(size)
            lw = 3 if label == 'Ladder' else 4
            color = '#444444' if label == 'Ladder' else '#e05c1a'
            ax.plot([x_center - 0.35, x_center + 0.35], [y, y],
                    color=color, linewidth=lw, solid_capstyle='butt')
            if label == 'Ladder':
                ax.text(x_center - 0.38, y, f'{size}', ha='right', va='center', fontsize=7)
            else:
                ax.text(x_center, y + 0.04, f'{size}bp', ha='center', va='bottom', fontsize=7)
        ax.text(x_center, 1.85, label, ha='center', va='top', fontsize=9, fontweight='bold')

    ax.set_title('Simulated Agarose Gel', fontsize=12)
    plt.tight_layout()
    plt.show()

lanes = {
    'EcoRI':        digest(plasmid_seq, 'EcoRI'),
    'BamHI':        digest(plasmid_seq, 'BamHI'),
    'EcoRI+BamHI':  double_digest(plasmid_seq, 'EcoRI', 'BamHI'),
}
plot_gel(lanes)

### 1.5 Compatible Ends: Which Enzymes Can Ligate Together?

Two restriction fragments can be ligated if their overhangs are complementary.
For 5' overhangs the single-stranded tail of one end must equal the reverse complement of the other.
Some enzymes produce **identical** overhangs even though their recognition sequences differ (e.g. BamHI/BglII both give 5'-GATC).

In [ ]:
def compute_overhang(enzyme_name: str) -> tuple[str, str]:
    """Return (overhang_sequence, overhang_type) for an enzyme.
    overhang_type is '5prime', '3prime', or 'blunt'.
    Overhang sequence is on the top strand for 5' overhangs,
    bottom strand (5'->3') for 3' overhangs.
    """
    site, cut_top, cut_bot = RESTRICTION_ENZYMES[enzyme_name]
    if cut_top == cut_bot:
        return ('', 'blunt')
    if cut_top < cut_bot:
        # 5' overhang: single-stranded portion is site[cut_top:cut_bot]
        return (site[cut_top:cut_bot], '5prime')
    else:
        # 3' overhang: single-stranded on bottom strand
        overhang = reverse_complement(site)[cut_bot: cut_top]
        return (overhang, '3prime')

def are_compatible(enzyme1: str, enzyme2: str) -> bool:
    """Return True if the ends produced by enzyme1 and enzyme2 can be ligated."""
    oh1, type1 = compute_overhang(enzyme1)
    oh2, type2 = compute_overhang(enzyme2)
    if type1 != type2:
        return False
    if type1 == 'blunt':
        return True  # all blunt ends are compatible
    return oh1 == oh2  # same overhang sequence = compatible

print('Overhang summary:')
for name in RESTRICTION_ENZYMES:
    oh, otype = compute_overhang(name)
    print(f'  {name:10s}  {otype:8s}  5\'-{oh}-3\'')

print('\nCompatibility pairs:')
enzymes = list(RESTRICTION_ENZYMES.keys())
for e1, e2 in itertools.combinations(enzymes, 2):
    if are_compatible(e1, e2) and e1 != e2:
        oh, _ = compute_overhang(e1)
        print(f'  {e1} + {e2}  (shared overhang: {oh})')

## 2. Primer Design

### 2.1 PCR Fundamentals

A PCR reaction uses two primers:
- **Forward primer**: matches the top strand, pointing right
- **Reverse primer**: matches the bottom strand, pointing left (i.e. its sequence is the reverse complement of the bottom strand)

Primer quality depends on several properties:

| Property | Recommended range |
|----------|-------------------|
| Length   | 18–25 bp |
| GC content | 40–60% |
| Tm       | 55–65 °C (matched between pair ±2 °C) |
| 3' end   | No runs of ≥4 identical bases |
| Self-complementarity | Low |

### 2.2 Tm Calculation: Three Models

In [ ]:
def gc_content(seq: str) -> float:
    seq = seq.upper()
    return (seq.count('G') + seq.count('C')) / len(seq)

def tm_basic(primer: str) -> float:
    """4+2 rule: Tm = 4*(G+C) + 2*(A+T). Only reliable for <20bp."""
    s = primer.upper()
    return 4 * (s.count('G') + s.count('C')) + 2 * (s.count('A') + s.count('T'))

def tm_salt_adjusted(primer: str, salt_mm: float = 50.0) -> float:
    """Salt-adjusted Tm (Wallace rule variant).
    Tm = 81.5 + 16.6*log10([Na+]) + 41*(GC) - 675/N
    salt_mm: [Na+] in mM.
    """
    n = len(primer)
    gc = gc_content(primer)
    return 81.5 + 16.6 * math.log10(salt_mm / 1000) + 41 * gc - 675 / n

# Nearest-neighbor thermodynamic parameters (SantaLucia 1998)
# dH in kcal/mol, dS in cal/mol/K
NN_PARAMS: dict[str, tuple[float, float]] = {
    'AA': (-7.9, -22.2), 'AT': (-7.2, -20.4), 'TA': (-7.2, -21.3), 'CA': (-8.5, -22.7),
    'GT': (-8.4, -22.4), 'CT': (-7.8, -21.0), 'GA': (-8.2, -22.2), 'CG': (-10.6, -27.2),
    'GC': (-9.8, -24.4), 'GG': (-8.0, -19.9), 'AC': (-7.8, -21.0), 'TC': (-8.2, -22.2),
    'AG': (-7.8, -21.0), 'TG': (-8.5, -22.7), 'TT': (-7.9, -22.2), 'CC': (-8.0, -19.9),
}
# Initiation parameters
NN_INIT_GC = (0.1, -2.8)   # dH, dS for GC terminal
NN_INIT_AT = (2.3, 4.1)    # dH, dS for AT terminal

def tm_nearest_neighbor(primer: str, dna_conc_nm: float = 250.0, salt_mm: float = 50.0) -> float:
    """Nearest-neighbor Tm using SantaLucia 1998 parameters.
    dna_conc_nm: total primer concentration in nM (assumes non-self-complementary).
    salt_mm: [Na+] in mM.
    """
    seq = primer.upper()
    R = 1.987  # cal/mol/K
    dH, dS = 0.0, 0.0
    # Initiation
    for end_base in (seq[0], seq[-1]):
        if end_base in 'GC':
            dH += NN_INIT_GC[0]; dS += NN_INIT_GC[1]
        else:
            dH += NN_INIT_AT[0]; dS += NN_INIT_AT[1]
    # Propagation
    for i in range(len(seq) - 1):
        dinuc = seq[i:i+2]
        h, s = NN_PARAMS.get(dinuc, (-8.0, -20.0))
        dH += h; dS += s
    # Salt correction (Owczarzy 2004 approximation)
    dS += 0.368 * (len(seq) - 1) * math.log(salt_mm / 1000)
    ct = dna_conc_nm * 1e-9  # mol/L
    tm_kelvin = (dH * 1000) / (dS + R * math.log(ct / 4)) - 273.15
    return tm_kelvin

# Compare all three models on example primers
test_primers = [
    ('gfp_fwd',  'ATGGTGAGCAAGGGCGAGGAG'),
    ('gfp_rev',  'TTACTTGTACAGCTCGTCCATGCC'),
    ('short',    'GCATGCAAGCTT'),
    ('long_at',  'AATAATAATAATAATAATAATAAT'),
]
print(f'{"Name":<12} {"Seq":<26} {"Tm_basic":>8} {"Tm_salt":>8} {"Tm_NN":>8}')
for name, seq in test_primers:
    print(f'{name:<12} {seq:<26} {tm_basic(seq):>8.1f} '
          f'{tm_salt_adjusted(seq):>8.1f} {tm_nearest_neighbor(seq):>8.1f}')

### 2.3 Automated Primer Design: Growing a Primer to Target Tm

In [ ]:
def design_primer(template: str, start: int, direction: str,
                  target_tm: float = 60.0, min_len: int = 18,
                  max_len: int = 28, tm_fn=tm_nearest_neighbor) -> dict:
    """Grow a primer from start position until target_tm is reached.
    direction: 'fwd' reads template left-to-right; 'rev' reads right-to-left
    and returns reverse complement.
    """
    best = None
    for length in range(min_len, max_len + 1):
        if direction == 'fwd':
            seq = template[start: start + length]
        else:
            seq = reverse_complement(template[start - length + 1: start + 1])
        if len(seq) < length:
            break
        tm = tm_fn(seq)
        gc = gc_content(seq)
        candidate = {'seq': seq, 'length': length, 'tm': tm, 'gc': gc, 'start': start}
        if best is None or abs(tm - target_tm) < abs(best['tm'] - target_tm):
            best = candidate
        if tm >= target_tm:
            break
    return best

# Target: amplify a region of pUC19 MCS
puc19_mcs = (
    'GAATTCGAGCTCGGTACCCGGGGATCCTCTAGAGTCGACCTGCAGGCATGCAAGCTT'
    'GGCGTAATCATGGTCATAGCTGTTTCCTGTGTGAAATTGTTATCCGCTCACAATTCC'
    'ACACAACATACGAGCCGGAAGCATAAAGTGTAAAGCCTGGGGTGCCTAATGAGTGAG'
)

fwd_primer = design_primer(puc19_mcs, start=0, direction='fwd', target_tm=60)
rev_primer = design_primer(puc19_mcs, start=len(puc19_mcs)-1, direction='rev', target_tm=60)

print('Forward primer:')
print(f'  Seq: {fwd_primer["seq"]}')
print(f'  Length: {fwd_primer["length"]} bp  Tm: {fwd_primer["tm"]:.1f}°C  GC: {fwd_primer["gc"]*100:.0f}%')
print('Reverse primer:')
print(f'  Seq: {rev_primer["seq"]}')
print(f'  Length: {rev_primer["length"]} bp  Tm: {rev_primer["tm"]:.1f}°C  GC: {rev_primer["gc"]*100:.0f}%')
print(f'  Tm difference: {abs(fwd_primer["tm"] - rev_primer["tm"]):.1f}°C')

### 2.4 Checking for Primer Dimers (3' Complementarity)

In [ ]:
def count_3prime_complementarity(primer1: str, primer2: str, check_len: int = 5) -> int:
    """Count matching bases in the last check_len bases of primer1 vs complement of primer2 3' end.
    A score >=3 indicates dimer risk.
    """
    tail1 = primer1[-check_len:].upper()
    tail2 = reverse_complement(primer2[-check_len:]).upper()
    return sum(a == b for a, b in zip(tail1, tail2))

def check_3prime_run(primer: str, run_len: int = 4) -> bool:
    """Return True if 3' end contains a homopolymer run of run_len or more."""
    tail = primer[-run_len:].upper()
    return len(set(tail)) == 1

def check_hairpin(primer: str, min_stem: int = 4, loop_min: int = 3) -> bool:
    """Simple hairpin check: look for stem-loop within the primer itself."""
    seq = primer.upper()
    n = len(seq)
    for stem in range(min_stem, n // 2):
        for i in range(n - 2 * stem - loop_min):
            left = seq[i: i + stem]
            right = reverse_complement(seq[i + stem + loop_min: i + 2 * stem + loop_min])
            if left == right:
                return True
    return False

# Evaluate primer pair
p1 = fwd_primer['seq']
p2 = rev_primer['seq']

dimer_score = count_3prime_complementarity(p1, p2)
self_dimer_fwd = count_3prime_complementarity(p1, p1)
self_dimer_rev = count_3prime_complementarity(p2, p2)

print('Primer QC report:')
print(f'  Fwd 3-prime run:       {check_3prime_run(p1)}')
print(f'  Rev 3-prime run:       {check_3prime_run(p2)}')
print(f'  Fwd hairpin:           {check_hairpin(p1)}')
print(f'  Rev hairpin:           {check_hairpin(p2)}')
print(f'  Fwd-Rev dimer score:   {dimer_score}/5  ({"RISK" if dimer_score >= 3 else "OK"})')
print(f'  Fwd self-dimer score:  {self_dimer_fwd}/5')
print(f'  Rev self-dimer score:  {self_dimer_rev}/5')

### 2.5 Adding Restriction Sites to Primers for Cloning

To clone a PCR product directionally, we append a restriction site (plus a short GC clamp and extra bases for efficient cutting at the very end of a linear fragment) to each primer's 5' end.

In [ ]:
def add_re_site_to_primer(primer: str, enzyme_name: str, clamp: str = 'GCGC') -> str:
    """Return primer with RE site prepended: clamp + recognition_site + primer."""
    site, _, _ = RESTRICTION_ENZYMES[enzyme_name]
    return clamp + site + primer

# Standard cloning strategy: BamHI on fwd, HindIII on rev
insert_seq = puc19_mcs[4:120]   # sub-region to clone

fwd_raw = design_primer(insert_seq, 0, 'fwd', target_tm=58)
rev_raw = design_primer(insert_seq, len(insert_seq)-1, 'rev', target_tm=58)

cloning_fwd = add_re_site_to_primer(fwd_raw['seq'], 'BamHI')
cloning_rev = add_re_site_to_primer(rev_raw['seq'], 'HindIII')

print('Cloning primers:')
print(f'  Fwd: 5\'-{cloning_fwd}-3\'')
print(f'       [{"GCGC"}][{RESTRICTION_ENZYMES["BamHI"][0]}][{fwd_raw["seq"]}]')
print(f'  Rev: 5\'-{cloning_rev}-3\'')
print(f'       [{"GCGC"}][{RESTRICTION_ENZYMES["HindIII"][0]}][{rev_raw["seq"]}]')
print(f'  Expected insert after digest: ~{len(insert_seq)} bp')

## 3. Cloning Design

### 3.1 Traditional Cloning: Vector + Insert with Compatible RE Sites

In restriction-enzyme cloning:
1. Both vector and insert are digested with the same enzyme(s)
2. Compatible ends are ligated
3. The insert reading frame must be continuous with the vector's start codon / tag

In [ ]:
# Minimal expression vector backbone (synthetic)
# Structure: promoter - [BamHI] - MCS - [HindIII] - terminator
vector_backbone = (
    'TTGACAATTAATCATCGGCTCGTATAATGTGTGG'   # promoter
    'GGATCC'                                # BamHI site
    'AAGCTT'                                # HindIII site  
    'AAAAAACCCCTTTTGGGGAATTC'               # downstream
)

# Insert: GFP coding sequence (abbreviated)
gfp_cds = (
    'ATGGTGAGCAAGGGCGAGGAGCTGTTCACCGGGGTGGTGCCCATCCTGGTCGAGCTGGACGGCGACGTAAAC'
    'GGCCACAAGTTCAGCGTGTCCGGCGAGGGCGAGGGCGATGCCACCTACGGCAAGCTGACCCTGAAGTTCATC'
    'TGCACCACCGGCAAGCTGCCCGTGCCCTGGCCCACCCTCGTGACCACCCTGACCTACGGCGTGCAGTGCTTC'
    'AGCCGCTACCCCGACCACATGAAGCAGCACGACTTCTTCAAGTCCGCCATGCCCGAAGGCTACGTCCAGGAG'
    'CGCACCATCTTCTTCAAGGACGACGGCAACTACAAGACCCGCGCCGAGGTGAAGTTCGAGGGCGACACCCTG'
    'GTGAACCGCATCGAGCTGAAGGGCATCGACTTCAAGGAGGACGGCAACATCCTGGGGCACAAGCTGGAGTAC'
    'AACTACAACAGCCACAACGTCTATATCATGGCCGACAAGCAGAAGAACGGCATCAAGGTGAACTTCAAGATC'
    'CGCCACAACATCGAGGACGGCAGCGTGCAGCTCGCCGACCACTACCAGCAGAACACCCCCATCGGCGACGGC'
    'CCCGTGCTGCTGCCCGACAACCACTACCTGAGCACCCAGTCCGCCCTGAGCAAAGACCCCAACGAGAAGCGC'
    'GATCACATGGTCCTGCTGGAGTTCGTGACCGCCGCCGGGATCACTCTCGGCATGGACGAGCTGTACAAGTAA'
)

def check_internal_re_sites(insert: str, enzymes: list[str]) -> dict[str, list[int]]:
    """Find any internal RE sites in the insert that would interfere with cloning."""
    return {e: find_cut_positions(insert, e) for e in enzymes
            if find_cut_positions(insert, e)}

cloning_enzymes = ['BamHI', 'HindIII']
internal_sites = check_internal_re_sites(gfp_cds, cloning_enzymes)
print(f'GFP insert length: {len(gfp_cds)} bp')
if internal_sites:
    print('WARNING - internal RE sites found:')
    for enz, positions in internal_sites.items():
        print(f'  {enz}: {positions}')
else:
    print('No internal BamHI or HindIII sites in GFP - safe for directional cloning.')

### 3.2 Verifying Reading Frame After Ligation

In [ ]:
CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def translate(cds: str) -> str:
    cds = cds.upper()
    return ''.join(CODON_TABLE.get(cds[i:i+3], '?') for i in range(0, len(cds)-2, 3))

def verify_reading_frame(vector: str, insert: str, upstream_enzyme: str) -> dict:
    """Check reading frame is preserved after ligating insert into vector.
    Assumes insert starts at the upstream enzyme cut site in the vector.
    """
    site_up, cut_top_up, _ = RESTRICTION_ENZYMES[upstream_enzyme]
    # Find ATG in vector before cut site
    vector_up = vector[:vector.find(site_up)]
    last_atg = vector_up.rfind('ATG')
    if last_atg == -1:
        return {'frame_ok': False, 'reason': 'No ATG found upstream'}
    # Distance from ATG to cut point
    cut_pos_in_vector = vector.find(site_up) + cut_top_up
    dist = cut_pos_in_vector - last_atg
    frame_offset = dist % 3
    protein = translate(insert)
    first_stop = protein.find('*')
    return {
        'frame_ok': frame_offset == 0,
        'frame_offset': frame_offset,
        'insert_protein_length': first_stop if first_stop != -1 else len(protein),
        'has_stop_codon': first_stop != -1,
        'insert_aa': protein[:10] + '...',
    }

# Simulate ligation: insert GFP after BamHI
result = verify_reading_frame(vector_backbone, gfp_cds, 'BamHI')
print('Reading frame check after ligation:')
for k, v in result.items():
    print(f'  {k}: {v}')

### 3.3 Simulating the Full Cloning Workflow

In [ ]:
def simulate_cloning(vector: str, insert: str,
                     upstream_enzyme: str, downstream_enzyme: str) -> dict:
    """Full cloning simulation: digest both, then ligate."""
    # Digest vector (take the largest fragment as linearised backbone)
    if upstream_enzyme == downstream_enzyme:
        vec_frags = digest(vector, upstream_enzyme)
    else:
        vec_frags = double_digest(vector, upstream_enzyme, downstream_enzyme)
    linearised_vector = max(vec_frags, key=len)

    # Digest insert PCR product
    if upstream_enzyme == downstream_enzyme:
        ins_frags = digest(insert, upstream_enzyme)
    else:
        ins_frags = double_digest(insert, upstream_enzyme, downstream_enzyme)
    # The insert is the piece between the two RE sites
    insert_frag = ins_frags[len(ins_frags)//2] if len(ins_frags) >= 3 else (
        ins_frags[-1] if ins_frags else insert)

    # Check compatible ends
    compat = are_compatible(upstream_enzyme, downstream_enzyme)

    # Construct recombinant (simplified: just concatenate with site scars)
    up_site = RESTRICTION_ENZYMES[upstream_enzyme][0]
    dn_site = RESTRICTION_ENZYMES[downstream_enzyme][0]
    recombinant = linearised_vector + up_site + insert_frag + dn_site

    return {
        'vector_fragments': sorted([len(f) for f in vec_frags], reverse=True),
        'insert_frag_size': len(insert_frag),
        'linearised_vector_size': len(linearised_vector),
        'recombinant_size': len(recombinant),
        'directional': upstream_enzyme != downstream_enzyme,
        'compatible_ends': compat if upstream_enzyme != downstream_enzyme else True,
    }

# Prepare insert with RE tails
insert_with_tails = (
    'GCGC' + RESTRICTION_ENZYMES['BamHI'][0] +
    gfp_cds +
    RESTRICTION_ENZYMES['HindIII'][0] + 'GCGC'
)

cloning_result = simulate_cloning(vector_backbone, insert_with_tails, 'BamHI', 'HindIII')
print('Cloning simulation result:')
for k, v in cloning_result.items():
    print(f'  {k}: {v}')

### 3.4 Gateway Cloning: attB/attP Recombination

Gateway cloning uses site-specific recombination between **attB** (on the PCR product) and **attP** (on the donor vector) sequences, catalysed by BP Clonase. The result is an **entry clone** carrying attL sites. A second LR reaction moves the insert into any destination vector.

Key attB sequences:
- attB1: `ACAAGTTTGTACAAAAAAGCAGGCT`
- attB2: `ACCACTTTGTACAAGAAAGCTGGGT`

In [ ]:
GATEWAY_SITES = {
    'attB1': 'ACAAGTTTGTACAAAAAAGCAGGCT',
    'attB2': 'ACCACTTTGTACAAGAAAGCTGGGT',
    'attP1': 'ACAAGTTTGTACAAAAAAGCAGGCTTC',
    'attP2': 'ACCACTTTGTACAAGAAAGCTGGGTT',
    'attL1': 'ACAAGTTTGTACAAAAAAGCAGGCTNN',
    'attL2': 'ACCACTTTGTACAAGAAAGCTGGGTNN',
    'attR1': 'ACAAGTTTGTACAAAAAAGCAGGCT',
    'attR2': 'ACCACTTTGTACAAGAAAGCTGGGT',
}

def design_gateway_primers(cds: str, add_kozak: bool = True) -> dict[str, str]:
    """Design attB1/attB2-flanked primers for Gateway BP reaction."""
    kozak = 'ACCATG' if add_kozak else ''
    # Forward: attB1 + optional Kozak + first 20bp of CDS (without start if kozak added)
    cds_start = cds[3:23] if (add_kozak and cds.startswith('ATG')) else cds[:20]
    fwd = GATEWAY_SITES['attB1'] + kozak + cds_start
    # Reverse: attB2 + reverse complement of last 20bp (including stop)
    rev = GATEWAY_SITES['attB2'] + reverse_complement(cds[-20:])
    return {
        'gateway_fwd': fwd,
        'gateway_rev': rev,
        'amplicon_size': len(cds) + len(GATEWAY_SITES['attB1']) + len(GATEWAY_SITES['attB2']),
    }

gw_primers = design_gateway_primers(gfp_cds)
print('Gateway primers for GFP:')
for k, v in gw_primers.items():
    if isinstance(v, str):
        print(f'  {k}: {v[:40]}... ({len(v)} bp)')
    else:
        print(f'  {k}: {v}')

## 4. CRISPR Guide RNA Design

### 4.1 CRISPR-Cas9 Mechanism

CRISPR-Cas9 uses a **guide RNA (gRNA)** that base-pairs with a 20 nt target sequence immediately upstream of a **PAM** (Protospacer Adjacent Motif). Cas9 cleaves 3 bp upstream of the PAM.

Different Cas orthologs recognise different PAMs:

| Nuclease | PAM | Cut | Notes |
|----------|-----|-----|-------|
| SpCas9   | NGG (3') | Blunt, –3 bp | Most widely used |
| SaCas9   | NNGRRT (3') | Blunt, –3 bp | Smaller, AAV-deliverable |
| AsCas12a | TTTV (5') | Staggered | PAM is 5' of target |

### 4.2 Finding PAM Sites and Extracting Guide Sequences

In [ ]:
CAS_CONFIGS = {
    'SpCas9': {
        'pam': 'NGG',
        'pam_position': '3prime',  # PAM is 3' of protospacer
        'guide_len': 20,
        'pam_regex': '[ATGC]GG',
    },
    'SaCas9': {
        'pam': 'NNGRRT',
        'pam_position': '3prime',
        'guide_len': 21,
        'pam_regex': '[ATGC][ATGC]G[AG][AG]T',
    },
    'Cas12a': {
        'pam': 'TTTV',
        'pam_position': '5prime',  # PAM is 5' of protospacer
        'guide_len': 23,
        'pam_regex': 'TTT[ACG]',
    },
}

def find_guides(target_dna: str, cas_name: str) -> list[dict]:
    """Find all candidate guide RNAs in target_dna for a given Cas nuclease.
    Searches both strands.
    Returns list of dicts with guide sequence, strand, position.
    """
    config = CAS_CONFIGS[cas_name]
    pam_regex = config['pam_regex']
    guide_len = config['guide_len']
    pam_pos = config['pam_position']
    guides = []

    for strand, seq in [('+', target_dna.upper()),
                         ('-', reverse_complement(target_dna))]:
        for m in re.finditer(f'(?={pam_regex})', seq):
            pam_start = m.start()
            if pam_pos == '3prime':
                guide_start = pam_start - guide_len
                if guide_start < 0:
                    continue
                guide_seq = seq[guide_start:pam_start]
                pam_seq = seq[pam_start: pam_start + len(config['pam'])]
            else:  # 5prime PAM (Cas12a)
                pam_end = pam_start + len(config['pam'])
                guide_end = pam_end + guide_len
                if guide_end > len(seq):
                    continue
                guide_seq = seq[pam_end:guide_end]
                pam_seq = seq[pam_start:pam_end]
            if len(guide_seq) == guide_len:
                guides.append({
                    'guide': guide_seq,
                    'pam': pam_seq,
                    'strand': strand,
                    'position': pam_start if strand == '+' else len(target_dna) - pam_start,
                    'cas': cas_name,
                })
    return guides

# Target: first exon of a hypothetical gene
target_gene = (
    'ATGGCCACCGAGCGCAGCGATCGCAACGGCTTCGGCATCGAGATGGCCCAGAAGATCCTGAAGGAG'
    'CTGAAGCAGCTGGACAACATCAACGGCGAGCGCATCGGCGTCAAGAACAAGGGCTTCGACGAGCTG'
    'ATCGAGGAGCTGCGCAAGCAGGGCATCGACAAGAACCGCATCCTGAAGGCCATCAACCAGTTCAAG'
    'CGCACCATCGAGGAGTACAAGAACGGCATGAACGCCATCGACGAGATCCGCAAGCAGTTCAAGAAG'
    'CTGGAGGAGCTGAAGGCCGGCAAGAAGGCCAAGGAGAAGCTGCAGGAGAAGATCATCGACCTGAAG'
)

spcas9_guides = find_guides(target_gene, 'SpCas9')
sacas9_guides = find_guides(target_gene, 'SaCas9')
cas12a_guides = find_guides(target_gene, 'Cas12a')

print(f'SpCas9 guides found:  {len(spcas9_guides)}')
print(f'SaCas9 guides found:  {len(sacas9_guides)}')
print(f'Cas12a guides found:  {len(cas12a_guides)}')
print('\nFirst 3 SpCas9 guides:')
for g in spcas9_guides[:3]:
    print(f'  {g["guide"]}  PAM={g["pam"]}  strand={g["strand"]}  pos={g["position"]}')

### 4.3 Scoring On-Target Efficiency

Guide RNA efficiency correlates with several sequence features (Doench et al. 2016 Rule Set 2 captures many, but here we implement a simplified scoring model):

- GC content 40–70%: optimal (outside this range → penalty)
- G at position 20 (PAM-proximal): slightly favoured
- No more than 4 consecutive T (RNA Pol III terminator signal)
- Avoid TTTT run anywhere in guide

In [ ]:
def score_guide_on_target(guide: str) -> float:
    """Return on-target score 0-1 based on sequence features."""
    seq = guide.upper()
    score = 1.0

    # GC content penalty
    gc = gc_content(seq)
    if gc < 0.40 or gc > 0.70:
        score -= 0.3
    elif gc < 0.45 or gc > 0.65:
        score -= 0.1

    # Poly-T run (RNA Pol III terminator)
    if 'TTTT' in seq:
        score -= 0.25

    # G at PAM-proximal end (position -1 relative to PAM)
    if seq[-1] == 'G':
        score += 0.05

    # Penalise starting with T (reduced transcription from U6 promoter)
    if seq[0] == 'T':
        score -= 0.1

    # Avoid homopolymer runs of 5+
    for base in 'ATGC':
        if base * 5 in seq:
            score -= 0.2

    return max(0.0, min(1.0, score))

def count_mismatches(seq1: str, seq2: str) -> int:
    return sum(a != b for a, b in zip(seq1.upper(), seq2.upper()))

def assess_off_targets(guide: str, genome_sample: str,
                       max_mismatches: int = 3) -> int:
    """Count potential off-target sites (both strands) with <=max_mismatches.
    genome_sample represents a genomic region to check.
    """
    guide_len = len(guide)
    count = 0
    for seq in [genome_sample.upper(), reverse_complement(genome_sample)]:
        for i in range(len(seq) - guide_len + 1):
            window = seq[i:i + guide_len]
            if count_mismatches(guide, window) <= max_mismatches:
                count += 1
    return count

# Simulate a partial genome for off-target checking
import random
random.seed(42)
simulated_genome = ''.join(random.choices('ATGC', k=2000))
# Embed 2 near-identical off-target sequences to make the demo realistic
guide_example = spcas9_guides[0]['guide']
near_miss = guide_example[:17] + 'AAA'  # 3 mismatches
simulated_genome = simulated_genome[:500] + near_miss + simulated_genome[500+len(near_miss):]

print(f'On-target scoring for first 5 SpCas9 guides:')
print(f'{"Guide":<22} {"GC%":>5} {"OnTarget":>9} {"OffTargets":>10} {"Rank"}')
scored_guides = []
for g in spcas9_guides[:8]:
    on_score = score_guide_on_target(g['guide'])
    off_count = assess_off_targets(g['guide'], simulated_genome)
    # Penalise off-targets: composite score
    composite = on_score * (1 / (1 + off_count * 0.2))
    scored_guides.append({**g, 'on_score': on_score,
                           'off_count': off_count, 'composite': composite})

scored_guides.sort(key=lambda x: x['composite'], reverse=True)
for rank, g in enumerate(scored_guides, 1):
    gc = gc_content(g['guide']) * 100
    print(f'{g["guide"][:20]:<22} {gc:>4.0f}% {g["on_score"]:>9.3f} '
          f'{g["off_count"]:>10d}  #{rank}')

### 4.4 Ranking the Top Guide RNAs

In [ ]:
def rank_guides(guides: list[dict], genome_sample: str) -> list[dict]:
    """Score, assess off-targets, and return guides sorted by composite score."""
    results = []
    for g in guides:
        on = score_guide_on_target(g['guide'])
        off = assess_off_targets(g['guide'], genome_sample)
        composite = on * (1 / (1 + off * 0.15))
        results.append({**g, 'on_score': on, 'off_count': off, 'composite': composite})
    return sorted(results, key=lambda x: x['composite'], reverse=True)

top_guides = rank_guides(spcas9_guides, simulated_genome)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# On-target score distribution
on_scores = [g['on_score'] for g in top_guides]
axes[0].hist(on_scores, bins=10, color='steelblue', edgecolor='white')
axes[0].set_xlabel('On-target score')
axes[0].set_ylabel('Number of guides')
axes[0].set_title(f'SpCas9 On-target Score Distribution (n={len(top_guides)})')

# Top 10 guides ranked
top10 = top_guides[:10]
labels = [g['guide'][:12]+'...' for g in top10]
composites = [g['composite'] for g in top10]
bars = axes[1].barh(range(len(top10)), composites, color='darkorange')
axes[1].set_yticks(range(len(top10)))
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel('Composite score')
axes[1].set_title('Top 10 SpCas9 Guides')

plt.tight_layout()
plt.show()

print('Top 3 guide RNAs for cloning:')
for i, g in enumerate(top_guides[:3], 1):
    print(f'  #{i}: 5\'-{g["guide"]}-3\' (strand {g["strand"]}, '
          f'on={g["on_score"]:.2f}, off={g["off_count"]})')

## 5. Codon Optimization

### 5.1 Why Codon Optimization Matters

The genetic code is **degenerate**: most amino acids are encoded by 2–6 synonymous codons. Each organism uses these codons at different frequencies (codon usage bias). When expressing a human gene in *E. coli*, rare-codon clusters cause ribosome stalling and reduced yield. Codon optimization replaces rare codons with the most frequently used synonymous codon in the host.

The **Codon Adaptation Index (CAI)** measures how well a CDS matches the host codon usage (range 0–1; values >0.8 are considered well-optimized).

In [ ]:
# Codon usage tables: relative synonymous codon usage (fraction of codon family)
# Data approximated from published genome-wide analyses
CODON_USAGE: dict[str, dict[str, dict[str, float]]] = {
    'ecoli': {
        'F': {'TTT': 0.51, 'TTC': 0.49},
        'L': {'TTA': 0.13, 'TTG': 0.13, 'CTT': 0.10, 'CTC': 0.10, 'CTA': 0.03, 'CTG': 0.50},
        'I': {'ATT': 0.51, 'ATC': 0.42, 'ATA': 0.07},
        'M': {'ATG': 1.00},
        'V': {'GTT': 0.26, 'GTC': 0.20, 'GTA': 0.15, 'GTG': 0.39},
        'S': {'TCT': 0.15, 'TCC': 0.15, 'TCA': 0.12, 'TCG': 0.13, 'AGT': 0.15, 'AGC': 0.28},
        'P': {'CCT': 0.16, 'CCC': 0.10, 'CCA': 0.19, 'CCG': 0.55},
        'T': {'ACT': 0.17, 'ACC': 0.40, 'ACA': 0.13, 'ACG': 0.27},
        'A': {'GCT': 0.18, 'GCC': 0.26, 'GCA': 0.21, 'GCG': 0.35},
        'Y': {'TAT': 0.57, 'TAC': 0.43},
        '*': {'TAA': 0.64, 'TAG': 0.07, 'TGA': 0.29},
        'H': {'CAT': 0.57, 'CAC': 0.43},
        'Q': {'CAA': 0.34, 'CAG': 0.66},
        'N': {'AAT': 0.45, 'AAC': 0.55},
        'K': {'AAA': 0.74, 'AAG': 0.26},
        'D': {'GAT': 0.63, 'GAC': 0.37},
        'E': {'GAA': 0.68, 'GAG': 0.32},
        'C': {'TGT': 0.45, 'TGC': 0.55},
        'W': {'TGG': 1.00},
        'R': {'CGT': 0.38, 'CGC': 0.40, 'CGA': 0.06, 'CGG': 0.10, 'AGA': 0.04, 'AGG': 0.02},
        'G': {'GGT': 0.34, 'GGC': 0.37, 'GGA': 0.11, 'GGG': 0.15},
    },
    'human': {
        'F': {'TTT': 0.45, 'TTC': 0.55},
        'L': {'TTA': 0.07, 'TTG': 0.13, 'CTT': 0.13, 'CTC': 0.20, 'CTA': 0.07, 'CTG': 0.40},
        'I': {'ATT': 0.36, 'ATC': 0.47, 'ATA': 0.17},
        'M': {'ATG': 1.00},
        'V': {'GTT': 0.18, 'GTC': 0.24, 'GTA': 0.11, 'GTG': 0.47},
        'S': {'TCT': 0.15, 'TCC': 0.22, 'TCA': 0.15, 'TCG': 0.06, 'AGT': 0.15, 'AGC': 0.24},
        'P': {'CCT': 0.28, 'CCC': 0.33, 'CCA': 0.27, 'CCG': 0.11},
        'T': {'ACT': 0.24, 'ACC': 0.36, 'ACA': 0.28, 'ACG': 0.12},
        'A': {'GCT': 0.26, 'GCC': 0.40, 'GCA': 0.23, 'GCG': 0.11},
        'Y': {'TAT': 0.43, 'TAC': 0.57},
        '*': {'TAA': 0.28, 'TAG': 0.20, 'TGA': 0.52},
        'H': {'CAT': 0.41, 'CAC': 0.59},
        'Q': {'CAA': 0.25, 'CAG': 0.75},
        'N': {'AAT': 0.46, 'AAC': 0.54},
        'K': {'AAA': 0.42, 'AAG': 0.58},
        'D': {'GAT': 0.46, 'GAC': 0.54},
        'E': {'GAA': 0.42, 'GAG': 0.58},
        'C': {'TGT': 0.45, 'TGC': 0.55},
        'W': {'TGG': 1.00},
        'R': {'CGT': 0.08, 'CGC': 0.19, 'CGA': 0.11, 'CGG': 0.21, 'AGA': 0.20, 'AGG': 0.20},
        'G': {'GGT': 0.16, 'GGC': 0.34, 'GGA': 0.25, 'GGG': 0.25},
    },
}

# Build reverse map: codon -> amino acid
CODON_TO_AA = {c: aa for aa, codons in CODON_TABLE.items()
               for c in [aa]}   # dummy; use CODON_TABLE directly

print('Codon usage tables loaded for:', list(CODON_USAGE.keys()))
print('Example - Leu codons in E. coli:')
for codon, freq in CODON_USAGE['ecoli']['L'].items():
    bar = '=' * int(freq * 30)
    print(f'  {codon}: {freq:.2f}  |{bar}')

### 5.2 Computing Codon Adaptation Index (CAI)

In [ ]:
def compute_cai(cds: str, organism: str) -> float:
    """Compute Codon Adaptation Index for a CDS against an organism's codon usage."""
    usage = CODON_USAGE[organism]
    log_w_sum = 0.0
    n_codons = 0
    cds = cds.upper()
    for i in range(0, len(cds) - 2, 3):
        codon = cds[i:i+3]
        aa = CODON_TABLE.get(codon)
        if aa is None or aa == '*':
            continue
        aa_usage = usage.get(aa, {})
        if not aa_usage:
            continue
        max_freq = max(aa_usage.values())
        codon_freq = aa_usage.get(codon, 1e-6)
        w = codon_freq / max_freq
        log_w_sum += math.log(w)
        n_codons += 1
    return math.exp(log_w_sum / n_codons) if n_codons > 0 else 0.0

# Human GFP vs E. coli usage
cai_human_in_ecoli = compute_cai(gfp_cds, 'ecoli')
cai_human_in_human = compute_cai(gfp_cds, 'human')
print(f'GFP CAI in E. coli: {cai_human_in_ecoli:.3f}  (original human-optimised sequence)')
print(f'GFP CAI in human:   {cai_human_in_human:.3f}')

### 5.3 Codon Optimization Algorithm

In [ ]:
def optimize_codons(cds: str, target_organism: str,
                    preserve_first_n_codons: int = 0) -> str:
    """Replace each codon with the most frequent synonymous codon in target_organism.
    preserve_first_n_codons: keep this many codons from the N-terminus unchanged
    (useful for preserving folding signals at domain boundaries).
    """
    usage = CODON_USAGE[target_organism]
    optimized = []
    cds = cds.upper()
    for codon_idx, i in enumerate(range(0, len(cds) - 2, 3)):
        codon = cds[i:i+3]
        aa = CODON_TABLE.get(codon)
        if aa is None or codon_idx < preserve_first_n_codons:
            optimized.append(codon)
            continue
        aa_usage = usage.get(aa, {})
        if not aa_usage:
            optimized.append(codon)
        else:
            best_codon = max(aa_usage, key=aa_usage.get)
            optimized.append(best_codon)
    return ''.join(optimized)

gfp_ecoli = optimize_codons(gfp_cds, 'ecoli')
cai_after = compute_cai(gfp_ecoli, 'ecoli')

# Verify protein sequence is preserved
original_protein = translate(gfp_cds)
optimized_protein = translate(gfp_ecoli)

print(f'Codon optimization: human GFP → E. coli expression')
print(f'  CAI before: {cai_human_in_ecoli:.3f}')
print(f'  CAI after:  {cai_after:.3f}')
print(f'  Protein sequence conserved: {original_protein == optimized_protein}')
print(f'  Nucleotide identity: {sum(a==b for a,b in zip(gfp_cds, gfp_ecoli))}/{len(gfp_cds)} '
      f'({100*sum(a==b for a,b in zip(gfp_cds, gfp_ecoli))/len(gfp_cds):.0f}%)')

### 5.4 Identifying Rare Codon Clusters

In [ ]:
def rare_codon_profile(cds: str, organism: str,
                       rare_threshold: float = 0.2) -> list[tuple[int, str, float]]:
    """Return (codon_index, codon, relative_usage) for all rare codons.
    rare_threshold: fraction of max usage below which a codon is 'rare'.
    """
    usage = CODON_USAGE[organism]
    rare = []
    cds = cds.upper()
    for idx, i in enumerate(range(0, len(cds) - 2, 3)):
        codon = cds[i:i+3]
        aa = CODON_TABLE.get(codon)
        if aa is None or aa == '*':
            continue
        aa_usage = usage.get(aa, {})
        if not aa_usage:
            continue
        max_freq = max(aa_usage.values())
        rel = aa_usage.get(codon, 0) / max_freq
        if rel <= rare_threshold:
            rare.append((idx, codon, rel))
    return rare

rare_in_ecoli = rare_codon_profile(gfp_cds, 'ecoli')
rare_in_ecoli_opt = rare_codon_profile(gfp_ecoli, 'ecoli')

# Plot relative codon usage along the GFP CDS
def get_per_codon_usage(cds: str, organism: str) -> list[float]:
    usage = CODON_USAGE[organism]
    scores = []
    cds = cds.upper()
    for i in range(0, len(cds) - 2, 3):
        codon = cds[i:i+3]
        aa = CODON_TABLE.get(codon)
        if aa is None or aa == '*':
            scores.append(1.0)
            continue
        au = usage.get(aa, {})
        max_f = max(au.values()) if au else 1.0
        scores.append(au.get(codon, 0) / max_f)
    return scores

orig_scores = get_per_codon_usage(gfp_cds, 'ecoli')
opt_scores  = get_per_codon_usage(gfp_ecoli, 'ecoli')

fig, ax = plt.subplots(figsize=(14, 4))
x = list(range(len(orig_scores)))
ax.fill_between(x, orig_scores, alpha=0.4, label='Original (human-codon)', color='tomato')
ax.fill_between(x, opt_scores,  alpha=0.4, label='Optimized (E. coli)',   color='steelblue')
ax.axhline(0.2, color='red', linestyle='--', linewidth=0.8, label='Rare codon threshold')
ax.set_xlabel('Codon position')
ax.set_ylabel('Relative codon usage (w)')
ax.set_title('Per-codon usage: GFP in E. coli before and after optimization')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Rare codons (w≤0.2) before: {len(rare_in_ecoli)}')
print(f'Rare codons (w≤0.2) after:  {len(rare_in_ecoli_opt)}')

## 6. Gibson Assembly Design

### 6.1 Gibson Assembly Principles

Gibson Assembly uses three enzymes in a single reaction:
1. **T5 exonuclease** — chews back 5' ends to expose single-stranded 3' overhangs
2. **Phusion polymerase** — fills in gaps after annealing of overlapping strands
3. **Taq ligase** — seals nicks

Each junction requires a **15–30 bp overlap** between adjacent fragments. The overlap sequences are added to primers as 5' tails.

In [ ]:
def design_gibson_overlaps(fragments: list[str],
                           overlap_len: int = 20) -> list[dict]:
    """Design Gibson Assembly primers for a list of ordered fragments.
    Returns primer pairs for each fragment.
    """
    n = len(fragments)
    primers = []
    for i, frag in enumerate(fragments):
        prev_frag = fragments[(i - 1) % n]
        next_frag = fragments[(i + 1) % n]

        # Forward primer: 3' end of previous fragment + start of this fragment
        overlap_fwd = prev_frag[-overlap_len:]
        annealing_fwd = frag[:20]
        fwd = overlap_fwd + annealing_fwd

        # Reverse primer: reverse complement of (end of this fragment + start of next)
        overlap_rev_rc = reverse_complement(next_frag[:overlap_len])
        annealing_rev = reverse_complement(frag[-20:])
        rev = overlap_rev_rc + annealing_rev

        # Check overlap GC content (ideally 40-60%)
        fwd_overlap_gc = gc_content(overlap_fwd)
        rev_overlap_gc = gc_content(next_frag[:overlap_len])

        primers.append({
            'fragment': i + 1,
            'fwd': fwd,
            'rev': rev,
            'fwd_overlap_gc': fwd_overlap_gc,
            'rev_overlap_gc': rev_overlap_gc,
            'fwd_tm': tm_nearest_neighbor(annealing_fwd),
            'rev_tm': tm_nearest_neighbor(reverse_complement(annealing_rev)),
        })
    return primers

# Three-fragment Gibson: promoter + GFP + terminator
promoter_frag = puc19_mcs[:80]
gfp_frag      = gfp_ecoli[:300]   # first 100 codons of optimized GFP
terminator_frag = (
    'GCTTTTTTGTTTTTTATGTGATGAGTATTATTCAGATCCCCCGTTTTCGCCCCGAAGCTGAAGACGATCAG'
    'CCGACGCATCGCATCCGAGCTCAAGCTTGAAGGGCGTTTTTTCCCGTTTTGAAACCGGCAAGCAAGAGC'
)

assembly_fragments = [promoter_frag, gfp_frag, terminator_frag]
gibson_primers = design_gibson_overlaps(assembly_fragments, overlap_len=20)

for p in gibson_primers:
    print(f'Fragment {p["fragment"]}:')
    print(f'  Fwd ({len(p["fwd"])}bp): 5\'-{p["fwd"][:30]}...-3\'')
    print(f'  Rev ({len(p["rev"])}bp): 5\'-{p["rev"][:30]}...-3\'')
    print(f'  Fwd overlap GC: {p["fwd_overlap_gc"]*100:.0f}%  '
          f'Annealing Tm fwd: {p["fwd_tm"]:.1f}°C  rev: {p["rev_tm"]:.1f}°C')

### 6.2 Verifying Overlap Quality

In [ ]:
def verify_gibson_assembly(fragments: list[str], overlap_len: int = 20) -> list[dict]:
    """Check that adjacent fragment overlaps are correct and flag issues."""
    reports = []
    n = len(fragments)
    for i in range(n):
        frag_a = fragments[i]
        frag_b = fragments[(i + 1) % n]
        junction_end   = frag_a[-overlap_len:]
        junction_start = frag_b[:overlap_len]
        gc = gc_content(junction_end)
        tm = tm_nearest_neighbor(junction_end)
        self_comp = check_hairpin(junction_end, min_stem=4)
        report = {
            'junction': f'F{i+1}→F{(i%n)+2}',
            'overlap_seq': junction_end,
            'gc': gc,
            'tm': tm,
            'hairpin_risk': self_comp,
            'gc_ok': 0.35 <= gc <= 0.65,
            'tm_ok': tm >= 48,
        }
        reports.append(report)
    return reports

junction_reports = verify_gibson_assembly(assembly_fragments)
print('Gibson Assembly junction QC:')
print(f'{"Junction":<10} {"GC%":>5} {"Tm":>6} {"GC_ok":>7} {"Tm_ok":>6} {"Hairpin":>8}')
for r in junction_reports:
    print(f'{r["junction"]:<10} {r["gc"]*100:>4.0f}% {r["tm"]:>6.1f} '
          f'{str(r["gc_ok"]):>7} {str(r["tm_ok"]):>6} {str(r["hairpin_risk"]):>8}')

## 7. Exercises

### Exercise 1: Design Cloning Primers for GFP with BamHI/HindIII Sites

Design a complete primer pair to PCR-amplify the full GFP CDS (`gfp_cds`) with BamHI appended to the forward primer and HindIII to the reverse primer. Use `tm_nearest_neighbor` with a target Tm of 62°C for the annealing portions. Report length, Tm, GC content, and dimer/hairpin status for each primer.

In [ ]:
# Exercise 1 solution
# Step 1: design annealing portions
ex1_fwd_raw = design_primer(gfp_cds, start=0, direction='fwd',
                             target_tm=62, tm_fn=tm_nearest_neighbor)
ex1_rev_raw = design_primer(gfp_cds, start=len(gfp_cds)-1, direction='rev',
                             target_tm=62, tm_fn=tm_nearest_neighbor)

# Step 2: add RE tails
ex1_fwd_cloning = add_re_site_to_primer(ex1_fwd_raw['seq'], 'BamHI')
ex1_rev_cloning = add_re_site_to_primer(ex1_rev_raw['seq'], 'HindIII')

# Step 3: QC
print('Exercise 1: GFP Cloning Primers')
print(f'Forward: 5\'-{ex1_fwd_cloning}-3\' ({len(ex1_fwd_cloning)} bp total)')
print(f'  Annealing: {ex1_fwd_raw["seq"]}  Tm={ex1_fwd_raw["tm"]:.1f}°C  GC={ex1_fwd_raw["gc"]*100:.0f}%')
print(f'Reverse: 5\'-{ex1_rev_cloning}-3\' ({len(ex1_rev_cloning)} bp total)')
print(f'  Annealing: {ex1_rev_raw["seq"]}  Tm={ex1_rev_raw["tm"]:.1f}°C  GC={ex1_rev_raw["gc"]*100:.0f}%')
print(f'Tm difference: {abs(ex1_fwd_raw["tm"] - ex1_rev_raw["tm"]):.1f}°C')
print(f'Dimer score: {count_3prime_complementarity(ex1_fwd_raw["seq"], ex1_rev_raw["seq"])}/5')
print(f'Fwd hairpin: {check_hairpin(ex1_fwd_raw["seq"])}  Rev hairpin: {check_hairpin(ex1_rev_raw["seq"])}')
print(f'Internal BamHI in GFP: {bool(find_cut_positions(gfp_cds, "BamHI"))}')
print(f'Internal HindIII in GFP: {bool(find_cut_positions(gfp_cds, "HindIII"))}')

### Exercise 2: Find and Rank CRISPR Guides for a Target Gene

Using `target_gene` defined earlier, find all SpCas9 guides, score them, and visualise the top 5 with their GC content and on-target scores.

In [ ]:
# Exercise 2 solution
ex2_guides = find_guides(target_gene, 'SpCas9')
ex2_ranked = rank_guides(ex2_guides, target_gene)  # use target gene itself as off-target check

print(f'Exercise 2: CRISPR Guide Ranking ({len(ex2_guides)} candidates)')
print(f'{"Rank":<5} {"Guide (20bp)":<22} {"Strand":>6} {"GC%":>5} {"OnScore":>8} {"OffCount":>9} {"Composite":>10}')
for rank, g in enumerate(ex2_ranked[:5], 1):
    print(f'#{rank:<4} {g["guide"]:<22} {g["strand"]:>6} '
          f'{gc_content(g["guide"])*100:>4.0f}% '
          f'{g["on_score"]:>8.3f} {g["off_count"]:>9d} {g["composite"]:>10.3f}')

# Bar chart of top 5
top5 = ex2_ranked[:5]
fig, ax = plt.subplots(figsize=(9, 3))
x = range(5)
width = 0.35
ax.bar([i - width/2 for i in x], [g['on_score'] for g in top5],
       width, label='On-target', color='steelblue')
ax.bar([i + width/2 for i in x], [g['composite'] for g in top5],
       width, label='Composite', color='darkorange')
ax.set_xticks(list(x))
ax.set_xticklabels([f'Guide #{i+1}' for i in x])
ax.set_ylabel('Score')
ax.set_title('Top 5 SpCas9 Guides: On-target vs Composite Score')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 3: Codon-Optimize a Human Gene for E. coli Expression

Take the first 150 bp of `gfp_cds` (representing a human gene fragment), compute CAI before and after optimization for E. coli, and plot the per-codon usage profile.

In [ ]:
# Exercise 3 solution
human_fragment = gfp_cds[:150]   # 50 codons
ecoli_fragment  = optimize_codons(human_fragment, 'ecoli')

cai_before = compute_cai(human_fragment, 'ecoli')
cai_after  = compute_cai(ecoli_fragment,  'ecoli')

orig_scores_ex3 = get_per_codon_usage(human_fragment, 'ecoli')
opt_scores_ex3  = get_per_codon_usage(ecoli_fragment,  'ecoli')

fig, ax = plt.subplots(figsize=(12, 3))
x3 = list(range(len(orig_scores_ex3)))
ax.plot(x3, orig_scores_ex3, 'o-', markersize=4, label=f'Human seq (CAI={cai_before:.3f})',
        color='tomato', alpha=0.8)
ax.plot(x3, opt_scores_ex3,  's-', markersize=4, label=f'E. coli opt (CAI={cai_after:.3f})',
        color='steelblue', alpha=0.8)
ax.axhline(0.2, color='red', linestyle='--', linewidth=0.8, label='Rare threshold (w=0.2)')
ax.set_xlabel('Codon position')
ax.set_ylabel('Relative codon usage (w)')
ax.set_title('Exercise 3: Per-codon usage before and after E. coli optimization')
ax.legend()
plt.tight_layout()
plt.show()

print(f'CAI before optimization: {cai_before:.3f}')
print(f'CAI after optimization:  {cai_after:.3f}')
print(f'Protein conserved: {translate(human_fragment) == translate(ecoli_fragment)}')

### Exercise 4: Design a 3-Fragment Gibson Assembly

Using a synthetic construct (promoter + codon-optimized GFP + terminator), design primers for a 3-fragment Gibson Assembly, verify junction quality, and report any issues.

In [ ]:
# Exercise 4 solution
ex4_promoter    = puc19_mcs[:60]
ex4_gfp_optim   = gfp_ecoli          # full E. coli-optimized GFP
ex4_terminator  = terminator_frag

ex4_fragments = [ex4_promoter, ex4_gfp_optim, ex4_terminator]
ex4_primers   = design_gibson_overlaps(ex4_fragments, overlap_len=25)
ex4_junctions = verify_gibson_assembly(ex4_fragments, overlap_len=25)

print('Exercise 4: 3-Fragment Gibson Assembly')
print(f'Fragment sizes: {[len(f) for f in ex4_fragments]} bp')
print(f'Total assembled size: {sum(len(f) for f in ex4_fragments)} bp (approx)\n')

print('Primers:')
for p in ex4_primers:
    print(f'  Fragment {p["fragment"]} Fwd ({len(p["fwd"])}bp Tm_ann={p["fwd_tm"]:.1f}°C): '
          f'{p["fwd"][:25]}...')
    print(f'  Fragment {p["fragment"]} Rev ({len(p["rev"])}bp Tm_ann={p["rev_tm"]:.1f}°C): '
          f'{p["rev"][:25]}...')

print('\nJunction QC:')
all_ok = True
for r in ex4_junctions:
    status = 'PASS' if (r['gc_ok'] and r['tm_ok'] and not r['hairpin_risk']) else 'WARN'
    if status == 'WARN':
        all_ok = False
    print(f'  {r["junction"]}: GC={r["gc"]*100:.0f}% Tm={r["tm"]:.1f}°C '
          f'Hairpin={r["hairpin_risk"]}  [{status}]')
print(f'\nOverall assembly design: {"READY FOR ORDERING" if all_ok else "REVIEW WARNINGS"}')

## Summary

| Topic | Key Functions | Key Concepts |
|-------|-------------|-------------|
| Restriction digestion | `find_cut_positions`, `digest`, `double_digest` | Recognition sites, overhang geometry |
| Gel simulation | `plot_gel` | Log-scale migration, banding patterns |
| Compatible ends | `compute_overhang`, `are_compatible` | 5'/3' overhangs, blunt ligation |
| Primer design | `design_primer`, `tm_nearest_neighbor` | 4+2 rule, salt-adjusted, nearest-neighbor Tm |
| Primer QC | `check_hairpin`, `count_3prime_complementarity` | Dimer risk, hairpin, 3' run |
| RE cloning | `simulate_cloning`, `verify_reading_frame` | Directional cloning, ORF continuity |
| Gateway cloning | `design_gateway_primers` | attB/attP recombination |
| CRISPR guides | `find_guides`, `score_guide_on_target` | PAM recognition, on/off-target scoring |
| Codon optimization | `optimize_codons`, `compute_cai` | Codon usage bias, CAI, rare clusters |
| Gibson Assembly | `design_gibson_overlaps`, `verify_gibson_assembly` | Overlap Tm, GC content, junction QC |

All algorithms in this notebook are implemented from first principles using only the Python standard library and `matplotlib` / `numpy` for visualisation.

---

[< Previous: Biochemistry and Enzyme Kinetics](../13_Biochemistry_and_Enzyme_Kinetics/02_enzyme_kinetics.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Population Genetics >](../15_Population_Genetics_and_Molecular_Evolution/01_population_genetics.ipynb)

---